# Задание 1

**В рамках данного задания я буду рассматривать равномерное распределение.
Начнем с того, что экспериментально проверим асимптотическую нормальность выборочного среднего, дисперсии и квантили порядка 0.5 (т.е. медианы) для равномерного распределения на интервале [0, 10].**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats


def draw_histogram(data, title, xlabel, ylabel, bins=30, xtarget=None, ytarget=None):
    plt.hist(data, bins, density=True, alpha=0.6, color='g')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if xtarget is not None and ytarget is not None:
        plt.plot(xtarget, ytarget, 'r--', linewidth=2)
    plt.grid()
    plt.show()


def transform(value, mean, count):
    return np.sqrt(count) * (value - mean)


values_count = [10, 100, 1000, 10000, 100000]
M = 3333
a = 0
b = 10
mean = (a + b) / 2
var = ((b - a) ** 2) / 12
median = (a + b) / 2

var_mean = (b - a) ** 2 / 12
var_variance = ((b - a) ** 4 / 80 - ((b - a) ** 2 / 12) ** 2)
var_median = (b - a) ** 2 / 4

vars = {
    "mean": var_mean,
    "variance": var_variance,
    "median": var_median
}
means = {
    "mean": mean,
    "variance": var,
    "median": median
}


def get_xtarget(name):
    return np.linspace(means[name] - 4 * np.sqrt(vars[name]), means[name] + 4 * np.sqrt(vars[name]), 1000)


data = {
    "mean": {},
    "variance": {},
    "median": {}}
np.random.seed(88005553)
for count in values_count:
    data["mean"][count] = []
    data["variance"][count] = []
    data["median"][count] = []
    for _ in range(M):
        sample = np.random.uniform(a, b, count)
        data["mean"][count].append(transform(np.mean(sample), mean, count))
        data["variance"][count].append(transform(np.var(sample), var, count))
        data["median"][count].append(transform(np.median(sample), median, count))

for item in data.items():
    name, values = item
    for n, data in values.items():
        draw_histogram(data, f"Asymptotic normality of the sample {name} for size {n}", name, "Density", bins=50,
                       xtarget=get_xtarget(name), ytarget=stats.norm.pdf(get_xtarget(name), 0, np.sqrt(vars[name])))

**Далее убедимся в том, что $nF(X_{(k)}) \to U_1 \sim \Gamma(k,1)$ и $n(1-F(X_{(n + 1 - j)})) \to U_2 \sim \Gamma(j,1)$**

In [ ]:
def f1(data, k, n):
    sorted = np.sort(data)
    X_k = sorted[k - 1]
    return n * stats.uniform.cdf(X_k, a, b - a)


def f2(data, j, n):
    sorted = np.sort(data)
    X_n_plus_1_minus_j = sorted[n - j]
    return n * (1 - stats.uniform.cdf(X_n_plus_1_minus_j, a, b - a))


k = 12
j = 1


def get_gamma_graph(k):
    x = np.linspace(0, max(15, k + 5 * np.sqrt(k)), 1000)
    y = stats.gamma.pdf(x, k, scale=1)
    return x, y


ns = [40, 200, 2000, 20000, 200000]
data1 = {}
data2 = {}
GEN_COUNT = 1000

for n in ns:
    data1[n] = []
    data2[n] = []
    for _ in range(GEN_COUNT):
        sample = np.random.uniform(a, b, n)
        data1[n].append(f1(sample, k, n))
        data2[n].append(f2(sample, j, n))

xtarget1, ytarget1 = get_gamma_graph(k)
xtarget2, ytarget2 = get_gamma_graph(j)
for (n, data1_n) in data1.items():
    draw_histogram(data1_n, f"Convergence of nF(X_(k)) to Gamma(k,1) for k={k} and n={n}", "nF(X_(k))", "Density", bins=50,xtarget=xtarget1, ytarget=ytarget1)

for (n, data2_n) in data2.items():
    draw_histogram(data2_n, f"Convergence of n(1-F(X_(n + 1 - j))) to Gamma(j,1) for j={j} and n={n}", "n(1-F(X_(n + 1 - j)))",
                   "Density", bins=50, xtarget=xtarget2, ytarget=ytarget2)





# Задание 2

Найдем дисперсию выборочного среднего и медианы (для медианы - асимптотическую):
### Асимптотическая дисперсия выборочной медианы: дельта-метод

Пусть $X_1,\ldots,X_n$ — независимые наблюдения из распределения

$$
F(x)=(1-\varepsilon)F_N(x;\mu,\sigma^2)
+\varepsilon F_N(x;\mu,\tau^2).
$$

Распределение симметрично относительно $\mu$, поэтому генеральная медиана равна $\mu$.

**1. Преобразование к равномерному распределению**

По интегральному преобразованию вероятностей $U_i=F(X_i)\sim U(0,1)$. Для нечётного объёма выборки $n=2k-1$:

$$
M_n=F^{-1}(U_{(k)}).
$$

Для порядковой статистики равномерной выборки:

$$
\mathbb E[U_{(k)}]=\frac12,\qquad
\operatorname{Var}(U_{(k)})=\frac{1}{4(n+2)}.
$$

Следовательно,

$$
\sqrt n\left(U_{(k)}-\frac12\right)
\xrightarrow{d}\mathcal N\left(0,\frac14\right).
$$

**2. Применение дельта-метода**

Положим $g(u)=F^{-1}(u)$. По дельта-методу:

$$
\sqrt n(M_n-\mu)
\xrightarrow{d}
\mathcal N\left(0,\frac{[g'(1/2)]^2}{4}\right).
$$

Из равенства $F(F^{-1}(u))=u$ следует:

$$
g'(u)=\frac{1}{f(F^{-1}(u))},
\qquad
g'\left(\frac12\right)=\frac1{f(\mu)}.
$$

Следовательно, асимптотическая дисперсия медианы равна

$$
\boxed{\operatorname{Var}(M_n)\approx\frac{1}{4n f^2(\mu)}}.
$$

**3. Подстановка плотности смеси**

В точке $\mu$ плотность составляет

$$
f(\mu)=\frac1{\sqrt{2\pi}}
\left(\frac{1-\varepsilon}{\sigma}
+\frac{\varepsilon}{\tau}\right).
$$

Таким образом,

$$
\boxed{
n\operatorname{Var}(M_n)\approx
\frac{\pi}{
2\left(\dfrac{1-\varepsilon}{\sigma}
+\dfrac{\varepsilon}{\tau}\right)^2
}
}.
$$

Формула является асимптотической и справедлива при $n\to\infty$ и $f(\mu)>0$.

### Дисперсия выборочного среднего

Пусть $X_1,\ldots,X_n$ — независимые одинаково распределённые случайные величины. Тогда

$$
\operatorname{Var}(\overline X)
=\frac{\operatorname{Var}(X_1)}{n}.
$$

Для смеси распределений

$$
F=(1-\varepsilon)N(\mu,\sigma^2)
+\varepsilon N(\mu,\tau^2)
$$

Математическое ожидание обеих компонент совпадает и равно $\mu$. Поэтому дисперсия смеси равна

$$
\operatorname{Var}(X_1)
=(1-\varepsilon)\sigma^2+\varepsilon\tau^2.
$$

Подставляя это выражение, получаем

$$
\boxed{
\operatorname{Var}(\overline X)
=\frac{(1-\varepsilon)\sigma^2+\varepsilon\tau^2}{n}
}.
$$

Следовательно,

$$
\boxed{
n\operatorname{Var}(\overline X)
=(1-\varepsilon)\sigma^2+\varepsilon\tau^2
}.
$$



In [ ]:


import pandas as pd

rng = np.random.default_rng(78798790)

mu = 0
sigma = 1
tau = 10
n = 100
N = 5000

eps_values = [0, 0.01, 0.05, 0.1, 0.2]

results = []
all_means = {}
all_medians = {}

for eps in eps_values:
    x = rng.normal(mu, sigma, size=(N, n))
    mask = rng.random((N, n)) < eps
    x[mask] = rng.normal(mu, tau, size=mask.sum())

    means = np.mean(x, axis=1)
    medians = np.median(x, axis=1)

    all_means[eps] = means
    all_medians[eps] = medians

    results.append({
        'Outlier proportion': eps,
        'Mean of sample means': means.mean(),
        'Mean of sample medians': medians.mean(),
        'Variance of means': means.var(ddof=1),
        'Variance of medians': medians.var(ddof=1),
        'SD of means': means.std(ddof=1),
        'SD of medians': medians.std(ddof=1)
    })

df = pd.DataFrame(results)
print(df.round(4).to_string(index=False))

In [ ]:
for eps in [0, 0.1]:
    means = all_means[eps]
    medians = all_medians[eps]

    mean_height = np.histogram(means, bins=30, density=True)[0].max()
    median_height = np.histogram(medians, bins=30, density=True)[0].max()

    draw_histogram(
        means,
        f'Sample Means, epsilon = {eps}',
        'Value',
        'Density',
        bins=30,
        xtarget=[mu, mu],
        ytarget=[0, mean_height]
    )

    draw_histogram(
        medians,
        f'Sample Medians, epsilon = {eps}',
        'Value',
        'Density',
        bins=30,
        xtarget=[mu, mu],
        ytarget=[0, median_height]
    )

In [ ]:
emp_mean = []
emp_median = []
theory_mean = []
theory_median = []

for eps in eps_values:
    emp_mean.append(n * np.var(all_means[eps], ddof=1))
    emp_median.append(n * np.var(all_medians[eps], ddof=1))

    theory_mean.append(
        (1 - eps) * sigma**2 + eps * tau**2
    )

    theory_median.append(
        np.pi / (
            2 * ((1 - eps) / sigma + eps / tau)**2
        )
    )

plt.figure(figsize=(10, 5))

plt.plot(eps_values, emp_mean, 'o-', label='Mean, empirical')
plt.plot(eps_values, theory_mean, 'o--', label='Mean, theoretical')
plt.plot(eps_values, emp_median, 's-', label='Median, empirical')
plt.plot(eps_values, theory_median, 's--', label='Median, theoretical')

plt.xlabel('Outlier Proportion')
plt.ylabel('n * Variance')
plt.title('Theoretical vs Empirical Variances')
plt.legend()
plt.grid()
plt.show()

In [ ]:
stats = pd.DataFrame({
    'Sample Mean': all_means[0.1],
    'Sample Median': all_medians[0.1]
})

print(stats.describe().round(4))

# Задание 3
датасет - data/bedtime_screen_sleep_debt.csv


In [ ]:


path = r"data/bedtime_screentime_sleep_debt.csv"

df = pd.read_csv(path)

feature_col = "bedtime_phone_minutes"
class_col = "sleep_debt_category"

data = df[[feature_col, class_col]].dropna()

print("Dataset shape:", data.shape)
print("\nNumber of observations by class:")
print(data[class_col].value_counts())

display(data.head())

In [ ]:
def descriptive_stats(x):
    x = np.asarray(x)

    q1 = np.quantile(x, 0.25)
    q3 = np.quantile(x, 0.75)

    return {
        "Count": len(x),
        "Mean": np.mean(x),
        "Sample variance": np.var(x, ddof=1),
        "Standard deviation": np.std(x, ddof=1),
        "Median": np.median(x),
        "Minimum": np.min(x),
        "Q1": q1,
        "Q3": q3,
        "Maximum": np.max(x),
        "Interquartile range": q3 - q1
    }


results = {
    "All data": descriptive_stats(data[feature_col])
}

for label, group in data.groupby(class_col):
    results[label] = descriptive_stats(group[feature_col])

stats_table = pd.DataFrame(results).T

display(stats_table.round(3))

In [ ]:
# All data
draw_histogram(
    data[feature_col],
    "Histogram — All Data",
    "Phone Usage Before Bedtime (Minutes)",
    "Density",
    bins=30
)

# Each class separately
for label, group in data.groupby(class_col):
    draw_histogram(
        group[feature_col],
        f"Histogram — {label}",
        "Phone Usage Before Bedtime (Minutes)",
        "Density",
        bins=30
    )

In [ ]:
def draw_ecdf(x, title, xlabel):
    x = np.sort(np.asarray(x))
    y = np.arange(1, len(x) + 1) / len(x)

    plt.figure(figsize=(8, 4))
    plt.step(x, y, where="post")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Empirical Cumulative Probability")
    plt.ylim(0, 1.02)
    plt.grid()
    plt.tight_layout()
    plt.show()


# All data
draw_ecdf(
    data[feature_col],
    "ECDF — All Data",
    "Phone Usage Before Bedtime (Minutes)"
)

# Each class separately
for label, group in data.groupby(class_col):
    draw_ecdf(
        group[feature_col],
        f"ECDF — {label}",
        "Phone Usage Before Bedtime (Minutes)"
    )

In [ ]:
# All data
plt.figure(figsize=(8, 3))
plt.boxplot(data[feature_col], orientation="horizontal")
plt.title("Box-plot — All Data")
plt.xlabel("Phone Usage Before Bedtime (Minutes)")
plt.yticks([1], ["All Data"])
plt.grid(axis="x")
plt.tight_layout()
plt.show()

# Each class separately
for label, group in data.groupby(class_col):
    plt.figure(figsize=(8, 3))
    plt.boxplot(group[feature_col], orientation="horizontal")
    plt.title(f"Box-plot — {label}")
    plt.xlabel("Phone Usage Before Bedtime (Minutes)")
    plt.yticks([1], [label])
    plt.grid(axis="x")
    plt.tight_layout()
    plt.show()

In [ ]:
# All data
plt.figure(figsize=(8, 4))
plt.violinplot(data[feature_col], showmedians=True)
plt.title("Violin Plot — All Data")
plt.ylabel("Phone Usage Before Bedtime (Minutes)")
plt.xticks([1], ["All Data"])
plt.grid(axis="y")
plt.tight_layout()
plt.show()

# Each class separately
for label, group in data.groupby(class_col):
    plt.figure(figsize=(8, 4))
    plt.violinplot(group[feature_col], showmedians=True)
    plt.title(f"Violin Plot — {label}")
    plt.ylabel("Phone Usage Before Bedtime (Minutes)")
    plt.xticks([1], [label])
    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()